In [3]:
import math
import numpy as np
import pandas as pd

LANGUAGE_ORDER = ["Python", "Java", "PHP", "JavaScript", "Ruby"]
HIGH_LEVEL = ["Symptom Category", "Root Cause Category", "Fix Pattern Category"]
ALL_LABELS = [
    "Symptom Category",
    "Symptom",
    "Root Cause Category",
    "Root Cause",
    "Fix Pattern Category",
    "Fix Pattern",
]

data = pd.read_excel("data/so_bugs.xlsx")
print("Records:", len(data))
print("Distinct labels:")
for c in ALL_LABELS:
    print(f"    {c}: data[c].nunique()")

Records: 1004
Distinct labels:
    Symptom Category: data[c].nunique()
    Symptom: data[c].nunique()
    Root Cause Category: data[c].nunique()
    Root Cause: data[c].nunique()
    Fix Pattern Category: data[c].nunique()
    Fix Pattern: data[c].nunique()


In [ ]:
def contingency(frame, group, label, group_order=None):
    table = pd.crosstab(frame[group], frame[label])
    return table.reindex(group_order) if group_order is not None else table


def distribution_table(frame, group, label, group_order=None):
    table = contingency(frame, group, label, group_order)
    percentages = table.div(table.sum(axis=1), axis=0) * 100
    formatted = table.astype(str) + " (" + percentages.map(lambda x: f"{x:.2f}%") + ")"
    formatted.index = [f"{name} (n={int(n)})" for name, n in table.sum(axis=1).items()]
    formatted.index.name = group
    return formatted.T

In [11]:
distribution_table(data, "Language", "Symptom Category", LANGUAGE_ORDER)

Language,Python (n=505),Java (n=164),PHP (n=123),JavaScript (n=169),Ruby (n=43)
Symptom Category,,,,,
Abnormal Rendering Result,252 (49.90%),75 (45.73%),47 (38.21%),89 (52.66%),26 (60.47%)
Compilation Error,113 (22.38%),41 (25.00%),33 (26.83%),38 (22.49%),13 (30.23%)
Initialization Error,33 (6.53%),22 (13.41%),8 (6.50%),13 (7.69%),1 (2.33%)
Others,10 (1.98%),4 (2.44%),2 (1.63%),3 (1.78%),1 (2.33%)
Placeholder Error,97 (19.21%),22 (13.41%),33 (26.83%),26 (15.38%),2 (4.65%)


In [12]:
distribution_table(data, "Language", "Root Cause Category", LANGUAGE_ORDER)

Language,Python (n=505),Java (n=164),PHP (n=123),JavaScript (n=169),Ruby (n=43)
Root Cause Category,,,,,
Improper Configuration,31 (6.14%),21 (12.80%),6 (4.88%),31 (18.34%),3 (6.98%)
Incompatible Integration,92 (18.22%),40 (24.39%),10 (8.13%),24 (14.20%),2 (4.65%)
Incorrect Property Resolution,62 (12.28%),17 (10.37%),13 (10.57%),15 (8.88%),1 (2.33%)
Mechanism Misconception,31 (6.14%),12 (7.32%),18 (14.63%),17 (10.06%),4 (9.30%)
Mismatched Data Context,109 (21.58%),16 (9.76%),30 (24.39%),29 (17.16%),11 (25.58%)
Syntax Misuse,180 (35.64%),58 (35.37%),46 (37.40%),53 (31.36%),22 (51.16%)


In [13]:
distribution_table(data, "Language", "Fix Pattern Category", LANGUAGE_ORDER)

Language,Python (n=505),Java (n=164),PHP (n=123),JavaScript (n=169),Ruby (n=43)
Fix Pattern Category,,,,,
Configuration-level Fix,43 (8.51%),33 (20.12%),16 (13.01%),19 (11.24%),2 (4.65%)
Host-side Fix,113 (22.38%),32 (19.51%),29 (23.58%),29 (17.16%),4 (9.30%)
Template-side Fix,349 (69.11%),99 (60.37%),78 (63.41%),121 (71.60%),37 (86.05%)


## Methods

All rows are retained as supplied, including repeated URLs with differing annotations.
Tests assume independent records. Engine selection is based on row count.
Distributions use within-language or within-engine percentages; leave-one-engine-out
distributions use the remaining sample size. Only high-level categories enter the tests.

Pearson's statistic is sum((observed - expected)^2 / expected), with no continuity
correction. Expected counts are row total × column total / N. The asymptotic method
is selected when no expected count is below 1 and at least 80% are at least 5.
Otherwise, category labels are permuted across records, preserving both margins.
Monte Carlo p = (exceedances + 1)/(permutations + 1). A zero-exceedance estimate is
the simulation resolution, not a measurement of the exact tail probability.

Cramér's V = sqrt(chi-square / (N × min(rows - 1, columns - 1))) is uncorrected.
For the tables here all degrees of freedom are even, so the asymptotic chi-square
survival probability is calculated with the exact finite-sum identity for even df.
SciPy is therefore not required. Reported test p-values are unadjusted; Holm results
are also supplied. These associations do not establish causal engine or language effects.


In [ ]:
N_PERMUTATIONS = 1_000_000
BATCH_SIZE = 2_000

def chi_square_even_df_survival(statistic, df):
    if df <= 0 or df % 2:
        raise ValueError("The finite-sum implementation requires positive even df.")
    z = statistic / 2
    return math.exp(-z) * sum(z**j / math.factorial(j) for j in range(df // 2))


def table_statistics(table):
    observed = table.to_numpy(dtype=np.int64)
    n = int(observed.sum())
    expected = np.outer(observed.sum(axis=1), observed.sum(axis=0)) / n
    statistic = float(((observed - expected) ** 2 / expected).sum())
    r, c = observed.shape
    df = (r - 1) * (c - 1)
    small = int((expected < 5).sum())
    tiny = int((expected < 1).sum())
    return dict(
        N=n,
        rows=r,
        columns=c,
        chi2=statistic,
        df=df,
        V=math.sqrt(statistic / (n * min(r - 1, c - 1))),
        min_expected=float(expected.min()),
        below5=small,
        below1=tiny,
        cells=expected.size,
        asymptotic_ok=tiny == 0 and small / expected.size <= 0.20,
        asymptotic_p=chi_square_even_df_survival(statistic, df),
    )


def permutation_test(frame, group, label, seed):
    # Alphabetical categorical encoding and input row order reproduce the original runs.
    groups = pd.Categorical(frame[group])
    categories = pd.Categorical(frame[label])
    r, c = len(groups.categories), len(categories.categories)
    observed = pd.crosstab(groups, categories).to_numpy()
    expected = np.outer(observed.sum(axis=1), observed.sum(axis=0)) / len(frame)
    statistic = float(((observed - expected) ** 2 / expected).sum())
    rng = np.random.default_rng(seed)
    group_base = groups.codes.astype(np.int64)[None, :] * c
    hits = 0
    for start in range(0, N_PERMUTATIONS, BATCH_SIZE):
        size = min(BATCH_SIZE, N_PERMUTATIONS - start)
        shuffled = rng.permuted(
            np.broadcast_to(categories.codes, (size, len(frame))), axis=1
        )
        offsets = (np.arange(size) * r * c)[:, None]
        tables = np.bincount(
            (offsets + group_base + shuffled).ravel(), minlength=size * r * c
        ).reshape(size, r, c)
        if start == 0:
            assert np.all(tables.sum(axis=2) == observed.sum(axis=1))
            assert np.all(tables.sum(axis=1) == observed.sum(axis=0))
        simulated = ((tables - expected) ** 2 / expected).sum(axis=(1, 2))
        hits += int(np.count_nonzero(simulated >= statistic - 1e-12))
    p = (hits + 1) / (N_PERMUTATIONS + 1)
    return dict(
        p=p,
        exceedances=hits,
        permutations=N_PERMUTATIONS,
        seed=seed,
        MCSE=math.sqrt(p * (1 - p) / N_PERMUTATIONS) if hits else float("nan"),
        zero_hit_upper95=(
            -math.expm1(math.log(0.05) / N_PERMUTATIONS) if hits == 0 else float("nan")
        ),
    )


def analyze(frame, group, permutation_seeds):
    results = {}
    for label in HIGH_LEVEL:
        result = table_statistics(contingency(frame, group, label))
        if result["asymptotic_ok"]:
            result.update(method="Asymptotic", p=result["asymptotic_p"])
        else:
            print("Running permutation test:", group, "×", label, flush=True)
            result.update(
                permutation_test(frame, group, label, permutation_seeds[label])
            )
            result["method"] = "Permutation"
        results[label] = result
    return results


def show_expected_checks(results):
    display(
        pd.DataFrame(
            {
                label: {
                    "Dimensions": f"{s['rows']} × {s['columns']}",
                    "Minimum expected": f"{s['min_expected']:.3f}",
                    "Expected <5": f"{s['below5']}/{s['cells']} ({100*s['below5']/s['cells']:.1f}%)",
                    "Expected <1": s["below1"],
                    "Method": s["method"],
                }
                for label, s in results.items()
            }
        ).T
    )


def show_tests(results):
    display(
        pd.DataFrame(
            {
                label: {
                    "Pearson chi-square": f"{s['chi2']:.3f}",
                    "df": s["df"],
                    "p-value": f"{s['p']:.8g}",
                    "Cramér's V": f"{s['V']:.4f}",
                    "Method": s["method"],
                }
                for label, s in results.items()
            }
        ).T
    )


def show_permutation_details(results):
    rows = {
        label: {
            k: s.get(k)
            for k in [
                "seed",
                "permutations",
                "exceedances",
                "p",
                "MCSE",
                "zero_hit_upper95",
            ]
        }
        for label, s in results.items()
        if s["method"] == "Permutation"
    }
    if rows:
        display(pd.DataFrame(rows).T)
    else:
        print(
            "All tables meet the expected-count guideline; no permutation tests required."
        )


def show_holm(results):
    labels = list(results)
    p = np.array([results[label]["p"] for label in labels])
    order = np.argsort(p)
    adjusted = np.empty(len(p))
    adjusted[order] = np.minimum(
        1, np.maximum.accumulate(p[order] * np.arange(len(p), 0, -1))
    )
    display(
        pd.DataFrame(
            {"Holm-adjusted p": adjusted, "Significant at .05": adjusted < 0.05},
            index=labels,
        )
    )

## Language-level analysis: all 1,004 records

In [ ]:
language_results = analyze(data, "Language", {"Symptom Category": 20261029})

Running permutation test: Language × Symptom Category


In [9]:
show_expected_checks(language_results)

,Dimensions,Minimum expected,Expected <5,Expected <1,Method
Symptom Category,5 × 5,0.857,5/25 (20.0%),1,Permutation
Root Cause Category,5 × 6,3.512,3/30 (10.0%),0,Asymptotic
Fix Pattern Category,5 × 3,4.840,1/15 (6.7%),0,Asymptotic


In [10]:
show_tests(language_results)

,Pearson chi-square,df,p-value,Cramér's V,Method
Symptom Category,29.504,16,0.021965978,0.0857,Permutation
Root Cause Category,72.026,20,8.4748381e-08,0.1339,Asymptotic
Fix Pattern Category,26.103,8,0.0010084873,0.1140,Asymptotic


In [11]:
show_permutation_details(language_results)

,seed,permutations,exceedances,p,MCSE,zero_hit_upper95
Symptom Category,20261029.0,1000000.0,21965.0,0.021966,0.000147,NaN


In [12]:
show_holm(language_results)

,Holm-adjusted p,Significant at .05
Symptom Category,2.196598e-02,True
Root Cause Category,2.542451e-07,True
Fix Pattern Category,2.016975e-03,True


## Engine-level analysis: engines with at least 20 records

In [ ]:
engine_counts = data["Template Engine"].value_counts()
eligible = engine_counts[engine_counts >= 20].index
engine_data = data[data["Template Engine"].isin(eligible)]
assert len(engine_data) == 961 and engine_data.URL.nunique() == 961
print("Retained:", len(engine_data), "records from", len(eligible), "engines")
display(engine_counts[engine_counts < 20].rename("Excluded records").to_frame())

Retained: 961 records from 9 engines


,Excluded records
Template Engine,
FreeMarker,14
ERB,9
Velocity,7
Mustache.js,6
Smarty,5
Haml,2


### Engine × Symptom Category: counts and within-engine percentages

In [ ]:
distribution_table(engine_data, "Template Engine", "Symptom Category")

Template Engine,Blade (n=72),Django-Template (n=327),EJS (n=97),Handlebars.js (n=33),Jinja (n=178),Liquid (n=32),Pug (n=33),Thymeleaf (n=143),Twig (n=46)
Symptom Category,,,,,,,,,
Abnormal Rendering Result,24 (33.3%),198 (60.6%),45 (46.4%),23 (69.7%),54 (30.3%),20 (62.5%),16 (48.5%),65 (45.5%),18 (39.1%)
Compilation Error,24 (33.3%),65 (19.9%),22 (22.7%),4 (12.1%),48 (27.0%),9 (28.1%),12 (36.4%),35 (24.5%),9 (19.6%)
Initialization Error,2 (2.8%),24 (7.3%),8 (8.2%),4 (12.1%),9 (5.1%),0 (0.0%),1 (3.0%),20 (14.0%),6 (13.0%)
Others,1 (1.4%),8 (2.4%),2 (2.1%),0 (0.0%),2 (1.1%),1 (3.1%),1 (3.0%),4 (2.8%),1 (2.2%)
Placeholder Error,21 (29.2%),32 (9.8%),20 (20.6%),2 (6.1%),65 (36.5%),2 (6.2%),3 (9.1%),19 (13.3%),12 (26.1%)


### Engine × Root Cause Category: counts and within-engine percentages

In [ ]:
distribution_table(engine_data, "Template Engine", "Root Cause Category")

Template Engine,Blade (n=72),Django-Template (n=327),EJS (n=97),Handlebars.js (n=33),Jinja (n=178),Liquid (n=32),Pug (n=33),Thymeleaf (n=143),Twig (n=46)
Root Cause Category,,,,,,,,,
Improper Configuration,3 (4.2%),25 (7.6%),17 (17.5%),8 (24.2%),6 (3.4%),1 (3.1%),6 (18.2%),19 (13.3%),3 (6.5%)
Incompatible Integration,4 (5.6%),76 (23.2%),13 (13.4%),5 (15.2%),16 (9.0%),0 (0.0%),6 (18.2%),37 (25.9%),6 (13.0%)
Incorrect Property Resolution,5 (6.9%),35 (10.7%),8 (8.2%),5 (15.2%),27 (15.2%),1 (3.1%),1 (3.0%),12 (8.4%),8 (17.4%)
Mechanism Misconception,13 (18.1%),13 (4.0%),7 (7.2%),4 (12.1%),18 (10.1%),3 (9.4%),4 (12.1%),9 (6.3%),4 (8.7%)
Mismatched Data Context,20 (27.8%),83 (25.4%),24 (24.7%),2 (6.1%),26 (14.6%),9 (28.1%),2 (6.1%),14 (9.8%),8 (17.4%)
Syntax Misuse,27 (37.5%),95 (29.1%),28 (28.9%),9 (27.3%),85 (47.8%),18 (56.2%),14 (42.4%),52 (36.4%),17 (37.0%)


### Engine × Fix Pattern Category: counts and within-engine percentages

In [ ]:
distribution_table(engine_data, "Template Engine", "Fix Pattern Category")

Template Engine,Blade (n=72),Django-Template (n=327),EJS (n=97),Handlebars.js (n=33),Jinja (n=178),Liquid (n=32),Pug (n=33),Thymeleaf (n=143),Twig (n=46)
Fix Pattern Category,,,,,,,,,
Configuration-level Fix,4 (5.6%),31 (9.5%),12 (12.4%),4 (12.1%),12 (6.7%),0 (0.0%),3 (9.1%),28 (19.6%),12 (26.1%)
Host-side Fix,20 (27.8%),94 (28.7%),19 (19.6%),5 (15.2%),19 (10.7%),2 (6.2%),3 (9.1%),30 (21.0%),7 (15.2%)
Template-side Fix,48 (66.7%),202 (61.8%),66 (68.0%),24 (72.7%),147 (82.6%),30 (93.8%),27 (81.8%),85 (59.4%),27 (58.7%)


In [ ]:
engine_results = analyze(
    engine_data,
    "Template Engine",
    {"Symptom Category": 20261029, "Root Cause Category": 20261030},
)

Running permutation test: Template Engine × Symptom Category
Running permutation test: Template Engine × Root Cause Category


In [18]:
show_expected_checks(engine_results)

,Dimensions,Minimum expected,Expected <5,Expected <1,Method
Symptom Category,9 × 5,0.666,12/45 (26.7%),4,Permutation
Root Cause Category,9 × 6,2.497,12/54 (22.2%),0,Permutation
Fix Pattern Category,9 × 3,3.530,3/27 (11.1%),0,Asymptotic


In [19]:
show_tests(engine_results)

,Pearson chi-square,df,p-value,Cramér's V,Method
Symptom Category,121.582,32,9.99999e-07,0.1778,Permutation
Root Cause Category,139.761,40,9.99999e-07,0.1705,Permutation
Fix Pattern Category,69.277,16,1.3353737e-08,0.1899,Asymptotic


In [20]:
show_permutation_details(engine_results)

,seed,permutations,exceedances,p,MCSE,zero_hit_upper95
Symptom Category,20261029.0,1000000.0,0.0,9.999990e-07,NaN,0.000003
Root Cause Category,20261030.0,1000000.0,0.0,9.999990e-07,NaN,0.000003


In [21]:
show_holm(engine_results)

,Holm-adjusted p,Significant at .05
Symptom Category,1.999998e-06,True
Root Cause Category,1.999998e-06,True
Fix Pattern Category,4.006121e-08,True


## Top-five-engine analysis: selected by record count

In [ ]:
top5_order = engine_counts.head(5).index.tolist()
top5_data = data[data["Template Engine"].isin(top5_order)]
assert len(top5_data) == 817 and top5_data.URL.nunique() == 817
display(engine_counts.head(5).rename("Records").to_frame())

,Records
Template Engine,
Django-Template,327
Jinja,178
Thymeleaf,143
EJS,97
Blade,72


### Top-five engines × Symptom Category: counts and within-engine percentages

In [ ]:
distribution_table(top5_data, "Template Engine", "Symptom Category", top5_order)

Template Engine,Django-Template (n=327),Jinja (n=178),Thymeleaf (n=143),EJS (n=97),Blade (n=72)
Symptom Category,,,,,
Abnormal Rendering Result,198 (60.6%),54 (30.3%),65 (45.5%),45 (46.4%),24 (33.3%)
Compilation Error,65 (19.9%),48 (27.0%),35 (24.5%),22 (22.7%),24 (33.3%)
Initialization Error,24 (7.3%),9 (5.1%),20 (14.0%),8 (8.2%),2 (2.8%)
Others,8 (2.4%),2 (1.1%),4 (2.8%),2 (2.1%),1 (1.4%)
Placeholder Error,32 (9.8%),65 (36.5%),19 (13.3%),20 (20.6%),21 (29.2%)


### Top-five engines × Root Cause Category: counts and within-engine percentages

In [ ]:
distribution_table(top5_data, "Template Engine", "Root Cause Category", top5_order)

Template Engine,Django-Template (n=327),Jinja (n=178),Thymeleaf (n=143),EJS (n=97),Blade (n=72)
Root Cause Category,,,,,
Improper Configuration,25 (7.6%),6 (3.4%),19 (13.3%),17 (17.5%),3 (4.2%)
Incompatible Integration,76 (23.2%),16 (9.0%),37 (25.9%),13 (13.4%),4 (5.6%)
Incorrect Property Resolution,35 (10.7%),27 (15.2%),12 (8.4%),8 (8.2%),5 (6.9%)
Mechanism Misconception,13 (4.0%),18 (10.1%),9 (6.3%),7 (7.2%),13 (18.1%)
Mismatched Data Context,83 (25.4%),26 (14.6%),14 (9.8%),24 (24.7%),20 (27.8%)
Syntax Misuse,95 (29.1%),85 (47.8%),52 (36.4%),28 (28.9%),27 (37.5%)


### Top-five engines × Fix Pattern Category: counts and within-engine percentages

In [ ]:
distribution_table(top5_data, "Template Engine", "Fix Pattern Category", top5_order)

Template Engine,Django-Template (n=327),Jinja (n=178),Thymeleaf (n=143),EJS (n=97),Blade (n=72)
Fix Pattern Category,,,,,
Configuration-level Fix,31 (9.5%),12 (6.7%),28 (19.6%),12 (12.4%),4 (5.6%)
Host-side Fix,94 (28.7%),19 (10.7%),30 (21.0%),19 (19.6%),20 (27.8%)
Template-side Fix,202 (61.8%),147 (82.6%),85 (59.4%),66 (68.0%),48 (66.7%)


In [ ]:
top5_results = analyze(top5_data, "Template Engine", {})

In [27]:
show_expected_checks(top5_results)

,Dimensions,Minimum expected,Expected <5,Expected <1,Method
Symptom Category,5 × 5,1.498,4/25 (16.0%),0,Asymptotic
Root Cause Category,5 × 6,5.288,0/30 (0.0%),0,Asymptotic
Fix Pattern Category,5 × 3,7.667,0/15 (0.0%),0,Asymptotic


In [28]:
show_tests(top5_results)

,Pearson chi-square,df,p-value,Cramér's V,Method
Symptom Category,93.880,16,4.7969035e-13,0.1695,Asymptotic
Root Cause Category,100.103,20,1.2070992e-12,0.1750,Asymptotic
Fix Pattern Category,43.021,8,8.7047528e-07,0.1623,Asymptotic


In [29]:
show_holm(top5_results)

,Holm-adjusted p,Significant at .05
Symptom Category,1.439071e-12,True
Root Cause Category,2.414198e-12,True
Fix Pattern Category,8.704753e-07,True


## Leave-one-engine-out sensitivity: full-dataset baseline

Each engine is removed separately from the original 1,004 records, not from the
top-five subset. Percentages use each run's remaining sample size. All observed
labels are retained. Label order follows whole-dataset frequency.


In [ ]:
runs = {"Whole dataset": data}
for engine in ["Django-Template", "Jinja", "Thymeleaf"]:
    runs[f"Without {engine}"] = data[data["Template Engine"] != engine]
display(
    pd.DataFrame(
        {
            "Records removed": {k: len(data) - len(v) for k, v in runs.items()},
            "Remaining records": {k: len(v) for k, v in runs.items()},
        }
    )
)

loo_percentages = {}
for label in ALL_LABELS:
    labels = data[label].value_counts().index
    counts = pd.DataFrame(
        {
            name: frame[label].value_counts().reindex(labels, fill_value=0)
            for name, frame in runs.items()
        }
    )
    assert counts.sum().to_dict() == {name: len(frame) for name, frame in runs.items()}
    percentages = (
        counts.div(
            pd.Series({name: len(frame) for name, frame in runs.items()}), axis=1
        )
        * 100
    )
    assert np.allclose(percentages.sum(), 100)
    loo_percentages[label] = percentages

,Records removed,Remaining records
Whole dataset,0,1004
Without Django-Template,327,677
Without Jinja,178,826
Without Thymeleaf,143,861


### Symptom Category: percentage distribution

In [ ]:
display(loo_percentages["Symptom Category"].map(lambda x: f"{x:.2f}"))

,Whole dataset,Without Django-Template,Without Jinja,Without Thymeleaf
Symptom Category,,,,
Abnormal Rendering Result,48.71,42.98,52.66,49.25
Compilation Error,23.71,25.55,23.00,23.58
Placeholder Error,17.93,21.86,13.92,18.70
Initialization Error,7.67,7.83,8.23,6.62
Others,1.99,1.77,2.18,1.86


### Symptom: percentage distribution

In [ ]:
display(loo_percentages["Symptom"].map(lambda x: f"{x:.2f}"))

,Whole dataset,Without Django-Template,Without Jinja,Without Thymeleaf
Symptom,,,,
Unexpected Output,26.69,26.88,26.76,26.71
Invalid Expression,15.24,16.84,15.25,13.94
Blank Output,11.25,7.53,13.08,11.96
Undefined Variable,7.67,10.64,4.36,8.83
Resource Not Found,7.57,7.24,9.08,7.55
Template Not Found,5.98,5.61,6.66,4.99
Property Access Error,5.48,7.09,5.33,4.76
Type Mismatch,4.78,4.14,4.24,5.11
Bad Delimiter,4.68,4.73,4.48,5.46


### Root Cause Category: percentage distribution

In [ ]:
display(loo_percentages["Root Cause Category"].map(lambda x: f"{x:.2f}"))

,Whole dataset,Without Django-Template,Without Jinja,Without Thymeleaf
Root Cause Category,,,,
Syntax Misuse,35.76,39.00,33.17,35.66
Mismatched Data Context,19.42,16.54,20.46,21.02
Incompatible Integration,16.73,13.59,18.40,15.21
Incorrect Property Resolution,10.76,10.78,9.81,11.15
Improper Configuration,9.16,9.90,10.41,8.48
Mechanism Misconception,8.17,10.19,7.75,8.48


### Root Cause: percentage distribution

In [ ]:
display(loo_percentages["Root Cause"].map(lambda x: f"{x:.2f}"))

,Whole dataset,Without Django-Template,Without Jinja,Without Thymeleaf
Root Cause,,,,
Expression Misuse,19.62,21.86,18.77,18.00
Inconsistent Data Type,13.84,11.96,13.80,15.10
Control Structure Misuse,10.76,11.67,9.32,11.38
Invalid Access Semantic,7.17,5.91,7.87,7.43
Incorrect Resource Reference,6.67,6.65,7.99,6.62
Template File Path Misconfiguration,5.88,5.32,6.78,4.99
Incomplete Data Context,5.58,4.58,6.66,5.92
Delimiter Misuse,5.38,5.47,5.08,6.27
Language Convention,4.48,5.61,4.12,4.30


### Fix Pattern Category: percentage distribution

In [ ]:
display(loo_percentages["Fix Pattern Category"].map(lambda x: f"{x:.2f}"))

,Whole dataset,Without Django-Template,Without Jinja,Without Thymeleaf
Fix Pattern Category,,,,
Template-side Fix,68.13,71.20,65.01,69.57
Host-side Fix,20.62,16.69,22.76,20.56
Configuration-level Fix,11.25,12.11,12.23,9.87


### Fix Pattern: percentage distribution

In [ ]:
display(loo_percentages["Fix Pattern"].map(lambda x: f"{x:.2f}"))

,Whole dataset,Without Django-Template,Without Jinja,Without Thymeleaf
Fix Pattern,,,,
Template Logic Modification,32.27,35.45,27.60,32.64
Template Syntax Correction,17.73,19.79,17.43,17.65
Data Context Refinement,13.94,10.19,15.25,14.52
Application Logic Refactoring,6.67,6.50,7.51,6.04
Property Resolution Correction,6.57,5.61,7.02,7.32
Resource Path Correction,6.18,6.06,7.38,6.16
Template Path Alignment,5.28,4.43,6.05,4.99
Engine Property Tuning,4.18,5.17,4.60,3.02
Template Logic Offloading,2.99,1.33,3.27,3.48


### Largest absolute change per distribution (percentage points)

In [ ]:
max_changes = pd.DataFrame(
    {
        label: percentages.iloc[:, 1:].sub(percentages.iloc[:, 0], axis=0).abs().max()
        for label, percentages in loo_percentages.items()
    }
).T
display(max_changes.map(lambda x: f"{x:.2f}"))

,Without Django-Template,Without Jinja,Without Thymeleaf
Symptom Category,5.72,4.01,1.05
Symptom,3.72,3.31,1.30
Root Cause Category,3.24,2.59,1.60
Root Cause,2.46,1.65,1.62
Fix Pattern Category,3.93,3.12,1.44
Fix Pattern,3.75,4.67,1.16


### Most frequent labels: stability across runs

In [ ]:
leaders = pd.DataFrame(
    {
        label: {
            name: frame[label].value_counts().index[0] for name, frame in runs.items()
        }
        for label in ALL_LABELS
    }
).T
display(leaders)
assert leaders.eq(leaders.iloc[:, 0], axis=0).all().all()

,Whole dataset,Without Django-Template,Without Jinja,Without Thymeleaf
Symptom Category,Abnormal Rendering Result,Abnormal Rendering Result,Abnormal Rendering Result,Abnormal Rendering Result
Symptom,Unexpected Output,Unexpected Output,Unexpected Output,Unexpected Output
Root Cause Category,Syntax Misuse,Syntax Misuse,Syntax Misuse,Syntax Misuse
Root Cause,Expression Misuse,Expression Misuse,Expression Misuse,Expression Misuse
Fix Pattern Category,Template-side Fix,Template-side Fix,Template-side Fix,Template-side Fix
Fix Pattern,Template Logic Modification,Template Logic Modification,Template Logic Modification,Template Logic Modification


## Total variation distance

For each distribution, compare whole-dataset probabilities P with each exclusion's
probabilities Q: D_TV(P,Q) = 0.5 × sum(abs(p_i - q_i)). Calculations use unrounded
probabilities. Parentheses express 100 × D_TV as the percentage of probability mass
that must be redistributed. Distances depend on both engine prevalence and composition.


In [ ]:
tv = pd.DataFrame(
    {
        label: percentages.iloc[:, 1:].sub(percentages.iloc[:, 0], axis=0).abs().sum()
        / 200
        for label, percentages in loo_percentages.items()
    }
).T
for label, percentages in loo_percentages.items():
    signed = percentages.iloc[:, 1:].sub(percentages.iloc[:, 0], axis=0) / 100
    assert np.allclose(tv.loc[label], signed.clip(lower=0).sum())
tv_formatted = tv.map(lambda x: f"{x:.5f} ({100*x:.2f}%)")
display(tv_formatted)

,Without Django-Template,Without Jinja,Without Thymeleaf
Symptom Category,0.05941 (5.94%),0.04709 (4.71%),0.01311 (1.31%)
Symptom,0.07139 (7.14%),0.04843 (4.84%),0.03421 (3.42%)
Root Cause Category,0.06022 (6.02%),0.03955 (3.95%),0.02304 (2.30%)
Root Cause,0.08068 (8.07%),0.05318 (5.32%),0.04569 (4.57%)
Fix Pattern Category,0.03926 (3.93%),0.03115 (3.12%),0.01443 (1.44%)
Fix Pattern,0.07512 (7.51%),0.05402 (5.40%),0.02410 (2.41%)


## Reproduction checks

These assertions check the reported numerical results, table totals, and original
permutation exceedance counts. Monte Carlo checks require the same random-number
behavior as the verified NumPy version; all nonrandom statistics are deterministic.


In [ ]:
expected_tests = {
    "language": [
        (29.504022290881476, 0.08571242089881156),
        (72.0260256295404, 0.13392075021456895),
        (26.10305439802358, 0.11401547738215662),
    ],
    "engine": [
        (121.58163735325542, 0.17784526397146472),
        (139.76144252677196, 0.17054814153186587),
        (69.27680446906592, 0.18985289840508796),
    ],
    "top5": [
        (93.87979864977767, 0.16949037899221123),
        (100.10332133972517, 0.17501820270756316),
        (43.02097657838695, 0.16226098433374708),
    ],
}
for name, results in [
    ("language", language_results),
    ("engine", engine_results),
    ("top5", top5_results),
]:
    for label, (chi, v) in zip(HIGH_LEVEL, expected_tests[name]):
        assert np.isclose(results[label]["chi2"], chi, rtol=1e-12)
        assert np.isclose(results[label]["V"], v, rtol=1e-12)
assert language_results["Symptom Category"]["exceedances"] == 21965
assert engine_results["Symptom Category"]["exceedances"] == 0
assert engine_results["Root Cause Category"]["exceedances"] == 0
expected_tv = np.array(
    [
        [0.0594093345966209, 0.04708526668145815, 0.013107847356219714],
        [0.07139095023157004, 0.048430973442790605, 0.03420695846116119],
        [0.06022439047355628, 0.039548826485824265, 0.023037929582483074],
        [0.08067876205664784, 0.053177122020393, 0.04569411089671512],
        [0.03926244799237321, 0.03115383502310367, 0.014427770914021039],
        [0.07512343535753585, 0.05401638000057883, 0.024098727043047323],
    ]
)
assert np.allclose(tv.to_numpy(), expected_tv, atol=1e-14, rtol=0)
print(
    "All reported chi-square statistics, effect sizes, permutation counts, and TV distances reproduced."
)

All reported chi-square statistics, effect sizes, permutation counts, and TV distances reproduced.
